# EpiScope Tutorial: Table Extraction and Reference Matching

This tutorial focuses on a key feature of the PrecisionMiner pipeline: extracting tables from PDF documents and linking their contents to the paper's bibliography.

**Prerequisites:**
1. An Ollama instance with a downloaded model (e.g., `ollama pull llama3:8b`).
2. Installation of `gmft` and `img2table` dependencies. You can install them with `pip install gmft img2table`.


## 1. Setup and Configuration

As in the other tutorials, we'll start by setting up our environment and creating a dummy PDF, this time with a table.


In [1]:
import os
from pathlib import Path
import warnings
import pandas as pd

# Suppress common warnings for a cleaner tutorial
warnings.filterwarnings('ignore', category=UserWarning)

# Define project paths
PDF_DIR = Path("./sample_pdfs")
PDF_DIR.mkdir(exist_ok=True)
PDF_PATH = PDF_DIR / "table_paper.pdf"

# Create a dummy PDF with a table
try:
    from reportlab.pdfgen import canvas
    from reportlab.lib.pagesizes import letter
    from reportlab.platypus import SimpleDocTemplate, Table, TableStyle
    from reportlab.lib import colors

    doc = SimpleDocTemplate(str(PDF_PATH), pagesize=letter)
    elements = []
    data = [['Author', 'Year', 'Finding'],
            ['Smith et al.', '2020', 'Finding A'],
            ['Jones et al.', '2021', 'Finding B']]
    t = Table(data)
    t.setStyle(TableStyle([('BACKGROUND', (0, 0), (-1, 0), colors.grey),
                           ('GRID', (0, 0), (-1, -1), 1, colors.black)]))
    elements.append(t)
    doc.build(elements)
    print(f"Created a dummy PDF with a table at: {PDF_PATH.resolve()}")
except ImportError:
    print("Please install reportlab (`pip install reportlab`) to create a dummy PDF.")
    print(f"Alternatively, place your own PDF with tables at: {PDF_PATH.resolve()}")


Please install reportlab (`pip install reportlab`) to create a dummy PDF.
Alternatively, place your own PDF with tables at: /Users/vins/Documents/Projects/EpiScope/episcope/notebooks/sample_pdfs/table_paper.pdf


## 2. Extracting Tables

The `TableExtractor` class is responsible for finding and extracting tables from a PDF. It can use multiple backends like `gmft` and `img2table`.


In [2]:
from episcope.parse.processing.figure_table_extractor import TableExtractor

OUTPUT_DIR = Path("./table_output")
table_extractor = TableExtractor(pdf_path=PDF_PATH, output_root=OUTPUT_DIR)

# The run_extractors method will use the available backends to find tables
extracted_tables = table_extractor.run_extractors()

if extracted_tables:
    print(f"Extracted {len(extracted_tables)} tables.")
    # Display the first table as a pandas DataFrame
    first_table_data = extracted_tables[0].get('data', [])
    if first_table_data:
        df = pd.DataFrame(first_table_data[1:], columns=first_table_data[0])
        print("\n--- First Extracted Table ---")
        print(df)
else:
    print("No tables were extracted. Check dependencies and PDF content.")


ModuleNotFoundError: No module named 'episcope'

## 3. Building and Matching Candidates

After extracting tables, the `TableExtractor` can build "candidate" strings from the table content. These candidates, which often contain author names or other identifiers, can then be matched against the paper's bibliography.


In [3]:
from episcope.parse.processing.figure_table_extractor import ReferenceMatcher
from episcope.core.blueprints.data_blueprints import Reference

# First, let's create some dummy references for our bibliography
references = [
    Reference(raw_text="Smith J, et al. (2020). A study on topic X. Journal of Science.", title="A study on topic X", authors=["Smith J"], year=2020),
    Reference(raw_text="Jones A, et al. (2021). An analysis of topic Y. Nature.", title="An analysis of topic Y", authors=["Jones A"], year=2021),
    Reference(raw_text="Williams B, et al. (2022). A review of Z. Cell.", title="A review of Z", authors=["Williams B"], year=2022)
]

# The run_with_matcher method automates the process of extraction, candidate building, and matching
matcher = ReferenceMatcher(matcher_backend="simple") # Use the lightweight surname matcher
match_results = table_extractor.run_with_matcher(references, matcher=matcher, extracted_tables=extracted_tables)

print("--- Match Results ---")
for result in match_results.get('comparison_results', []):
    print(f"Candidate: '{result['candidate_text']}'")
    if result['has_match']:
        print(f"  => Matched: '{result['reference_title']}' (Score: {result['match_score']:.2f})\n")
    else:
        print("  => No match found.\
")


ModuleNotFoundError: No module named 'episcope'

## Conclusion

This notebook demonstrated how to use the `TableExtractor` and `ReferenceMatcher` to extract structured data from tables within a PDF and link it to bibliographic references. This powerful feature enables EpiScope to understand the connections between a paper's findings (often summarized in tables) and the broader scientific literature.
